##Fine-tuning FinBERT for POS Tagging in Low Resource Uralic Languages

In [ ]:
!pip install transformers torch -q

In [ ]:
import json
import random
import os
from collections import defaultdict
from datetime import datetime
import numpy as np
import torch
from torch.optim import AdamW
from torch.utils.data import Dataset
from sklearn.metrics import precision_recall_fscore_support
from transformers import (
    AutoModelForTokenClassification,
    AutoTokenizer,
    Trainer,
    set_seed,
    TrainingArguments,
    DataCollatorForTokenClassification,
    EarlyStoppingCallback,
)


In [ ]:
FINBERT = "TurkuNLP/bert-base-finnish-cased-v1"
OUTPUT_DIR = "./pos-tagging"
RESULTS_DIR = "./results"
os.makedirs(RESULTS_DIR, exist_ok = True)

LANGUAGE selection -> 'sme' for North Sámi, 'myv' for Erzya

STRAT selection -> 'zero-shot' for baseline, 'ft' for fine-tuning
(STRAT parameter is only used for naming the output file, and is not needed elsewhere).

In [ ]:
LANGUAGE = "sme"
STRAT = "ft"

In [ ]:
LR = 5e-5
EPOCHS = 50
BATCH_SIZE = 16
EARLY_STOPPING_PATIENCE = 5
EARLY_STOPPING_THRESHOLD = 0.0001
METRIC = "accuracy"
MAX_LENGTH = 128

EXCLUDED_TAGS = {"PUNCT"}

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
TRAIN_FILE = "sme_giella-ud-train.conllu"
TEST_FILE = "sme_giella-ud-test.conllu"
TRAIN_OUT_FILE = "sme-train.conllu"
DEV_OUT_FILE = "sme-dev.conllu"
DEV_SIZE = 0.1

In [ ]:
SEED = 7
set_seed(SEED)

##Preprocessing

In [ ]:
def read_raw(path):
    sent, current = [], []
    with open(path, encoding="utf-8") as f:
        for line in f:
            current.append(line)
            if line.strip() == "":
                sent.append(current)
                current = []
    return sent


def write_raw(sent, path):
    with open(path, "w", encoding="utf-8") as f:
        for sen in sent:
            f.writelines(sen)


sent = read_raw(TRAIN_FILE)
random.seed(SEED)
random.shuffle(sent)

dev_num = int(len(sent) * DEV_SIZE)
dev_sent = sent[:dev_num]
train_sent = sent[dev_num:]

write_raw(train_sent, TRAIN_OUT_FILE)
write_raw(dev_sent, DEV_OUT_FILE)

print(f"Total: {len(sent)}")
print(f"Train: {len(train_sent)}")
print(f"Dev:   {dev_num}")

In [ ]:
def read_conllu(path):
    sentences = []
    words, tags = [], []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                if words:
                    sentences.append((words, tags))
                    words, tags = [], []
            else:
                cols = line.split("\t")
                if "-" in cols[0] or "." in cols[0]:
                    continue
                words.append(cols[1])
                tags.append(cols[3]) #upos
    return sentences


def build_tag2id(sentences):
    tags = sorted(set(tag for _, sent_tags in sentences for tag in sent_tags))
    return {tag: i for i, tag in enumerate(tags)}

def tokenize_and_align(sentences, tokenizer, tag2id):
    encoded = []
    for words, tags in sentences:
        tokenized = tokenizer(
            words,
            is_split_into_words=True,
            truncation=True,
            padding="max_length",
            max_length=MAX_LENGTH,
        )
        labels = []
        prev_word_id = None
        for word_id in tokenized.word_ids():
            if word_id is None:
                labels.append(-100)
            elif word_id != prev_word_id:
                labels.append(tag2id[tags[word_id]])
            else:
                labels.append(-100)
            prev_word_id = word_id
        tokenized["labels"] = labels
        encoded.append(tokenized)
    return encoded


class POSDataset(Dataset):
    def __init__(self, encoded):
        self.encoded = encoded

    def __len__(self):
        return len(self.encoded)

    def __getitem__(self, idx):
        item = self.encoded[idx]
        return {
            "input_ids":      torch.tensor(item["input_ids"]),
            "attention_mask": torch.tensor(item["attention_mask"]),
            "token_type_ids": torch.tensor(item["token_type_ids"]),
            "labels":         torch.tensor(item["labels"]),
        }

In [ ]:
train_sentences = read_conllu(TRAIN_OUT_FILE)
dev_sentences   = read_conllu(DEV_OUT_FILE)
test_sentences  = read_conllu(TEST_FILE)

tag2id = build_tag2id(test_sentences)
id2tag = {i: t for t, i in tag2id.items()}
print(f"  Tags ({len(tag2id)}): {', '.join(sorted(tag2id))}")

EXCLUDE_TAG_IDS = {tag2id[t] for t in EXCLUDED_TAGS if t in tag2id}
tokenizer     = AutoTokenizer.from_pretrained(FINBERT)
train_dataset = POSDataset(tokenize_and_align(train_sentences, tokenizer, tag2id))
dev_dataset   = POSDataset(tokenize_and_align(dev_sentences,   tokenizer, tag2id))
test_dataset  = POSDataset(tokenize_and_align(test_sentences,  tokenizer, tag2id))

##Fine-tuning

In [ ]:
model = AutoModelForTokenClassification.from_pretrained(
    FINBERT, num_labels=len(tag2id), id2label=id2tag, label2id=tag2id,
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    mask = labels != -100
    for excluded_id in EXCLUDE_TAG_IDS:
        mask &= labels != excluded_id

    flat_preds  = predictions[mask]
    flat_labels = labels[mask]

    accuracy = (flat_preds == flat_labels).sum() / len(flat_labels)
    precision, recall, f1, _ = precision_recall_fscore_support(
        flat_labels, flat_preds, average="macro", zero_division=0,
    )
    return {
        "accuracy":  float(accuracy),
        "precision": float(precision),
        "recall":    float(recall),
        "f1":        float(f1),
    }

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    learning_rate=LR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    eval_strategy="epoch",
    save_strategy="epoch",
    seed=SEED,
    load_best_model_at_end=True,
    metric_for_best_model=METRIC,
    logging_steps=50,
)

callbacks = []
if EARLY_STOPPING_PATIENCE is not None:
    callbacks.append(EarlyStoppingCallback(
        early_stopping_patience=EARLY_STOPPING_PATIENCE,
        early_stopping_threshold=EARLY_STOPPING_THRESHOLD,
    ))

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    data_collator=DataCollatorForTokenClassification(tokenizer),
    compute_metrics=compute_metrics,
    callbacks=callbacks,
)

trainer.train()

In [ ]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
with open(f"{OUTPUT_DIR}/tag2id.json", "w", encoding="utf-8") as f:
    json.dump({"tag2id": tag2id}, f, ensure_ascii=False, indent=2)

print(f"Model saved to {OUTPUT_DIR}")

Prediction

In [ ]:
output      = trainer.predict(test_dataset)
predictions = np.argmax(output.predictions, axis=-1)
labels      = output.label_ids


flat_preds, flat_labels = [], []
for pred_seq, label_seq in zip(predictions, labels):
    for p, l in zip(pred_seq, label_seq):
        if l != -100:
            flat_preds.append(id2tag[p])
            flat_labels.append(id2tag[l])

##Zero-shot Classification

In [ ]:
from transformers import pipeline
from tqdm.auto import tqdm

In [ ]:
candidate_labels = list(tag2id.keys())
classifier = pipeline(
    "zero-shot-classification",
    model= FINBERT,
)


flat_preds, flat_labels = [], []

for words, gold_tags in tqdm(test_sentences, desc="Sentences"):
    sentence_text = " ".join(words)
    for word, gold in zip(words, gold_tags):
        hypothesis_template = f"{word} is a {{}}."
        result = classifier(sentence_text, candidate_labels, hypothesis_template=hypothesis_template)
        flat_preds.append(result['labels'][0])
        flat_labels.append(gold)


##Evaluation

In [ ]:
tag_correct = defaultdict(int)
tag_total   = defaultdict(int)

for pred, gold in zip(flat_preds, flat_labels):
    tag_total[gold] += 1
    if pred == gold:
        tag_correct[gold] += 1

per_tag_accuracy = {
    tag: (tag_correct[tag] / tag_total[tag] if tag_total[tag] > 0 else None)
    for tag in sorted(tag2id.keys())
}


eval_preds, eval_labels = [], []
for pred, gold in zip(flat_preds, flat_labels):
    if gold not in EXCLUDED_TAGS:
        eval_preds.append(pred)
        eval_labels.append(gold)

overall_correct  = sum(1 for p, g in zip(eval_preds, eval_labels) if p == g)
overall_total    = len(eval_labels)
overall_accuracy = overall_correct / overall_total


all_tags      = sorted(tag2id.keys())
scored_tags   = [t for t in all_tags if t not in EXCLUDED_TAGS]

per_tag_precision, per_tag_recall, per_tag_f1, per_tag_support = precision_recall_fscore_support(
    flat_labels, flat_preds, labels=all_tags, zero_division=0,
)
per_tag_precision = dict(zip(all_tags, per_tag_precision))
per_tag_recall    = dict(zip(all_tags, per_tag_recall))
per_tag_f1        = dict(zip(all_tags, per_tag_f1))

macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(
    eval_labels, eval_preds, labels=scored_tags, average="macro", zero_division=0,
)
weighted_precision, weighted_recall, weighted_f1, _ = precision_recall_fscore_support(
    eval_labels, eval_preds, labels=scored_tags, average="weighted", zero_division=0,
)

if EXCLUDED_TAGS:
    print(f"Excluded from overall/macro/weighted scores: {sorted(EXCLUDED_TAGS & set(tag2id))}\n")
print(f"Overall accuracy:  {overall_accuracy:.4f} ({overall_correct}/{overall_total} tokens)")
print(f"Macro precision: {macro_precision:.4f} | recall: {macro_recall:.4f} | f1: {macro_f1:.4f}")
print(f"Weighted precision: {weighted_precision:.4f} | recall: {weighted_recall:.4f} | f1: {weighted_f1:.4f}\n")

print(f"{'Tag':<12} {'Correct':>8} {'Total':>8} {'Accuracy':>10} {'Precision':>11} {'Recall':>9} {'F1':>8}")
print("-" * 72)
for tag in all_tags:
    excluded_note = " (excluded)" if tag in EXCLUDED_TAGS else ""
    if tag_total[tag] == 0:
        print(f"{tag:<12} {'N/A':>8} {'0':>8} {'N/A':>10} {'N/A':>11} {'N/A':>9} {'N/A':>8}{excluded_note}")
    else:
        print(
            f"{tag:<12} {tag_correct[tag]:>8} {tag_total[tag]:>8} "
            f"{per_tag_accuracy[tag]:>10.4f} {per_tag_precision[tag]:>11.4f} "
            f"{per_tag_recall[tag]:>9.4f} {per_tag_f1[tag]:>8.4f}{excluded_note}"
        )

annotated_sentences = []
pred_idx = 0
for words, gold_tags in test_sentences:
    annotated = []
    for word, gold in zip(words, gold_tags):
        predicted = flat_preds[pred_idx]
        annotated.append((word, gold, predicted))
        pred_idx += 1
    annotated_sentences.append(annotated)

In [ ]:
STOPPED_EPOCH = 8 #manually update according to the training process
NOTES = " " #any additional notes about the process 

In [ ]:
timestamp   = datetime.now().strftime("%Y%m%d_%H%M%S")
result_path = f"{RESULTS_DIR}/{LANGUAGE}_{STRAT}_{timestamp}.json"
annot_path  = f"{RESULTS_DIR}/{LANGUAGE}_{STRAT}_{timestamp}_annotated.conllu"

with open(annot_path, "w", encoding="utf-8") as f:
    #save the parameters (LR and early stopping parameters are commented out for zero-shot baseline)
    f.write(f"# language: {LANGUAGE}\n")
    f.write(f"# early stopped at: {STOPPED_EPOCH}\n")
    f.write(f"# model: {FINBERT}\n")
    f.write(f"# strategy: {STRAT}\n")
    #f.write(f"# lr: {LR} | epochs: {EPOCHS} | batch_size: {BATCH_SIZE}\n")
    #f.write(f"# early_stopping_patience: {EARLY_STOPPING_PATIENCE} | metric_for_best_model: {METRIC_FOR_BEST_MODEL}\n")
    f.write(f"# overall_accuracy: {overall_accuracy:.4f}\n")
    f.write(f"# macro_f1: {macro_f1:.4f} | macro_precision: {macro_precision:.4f} | macro_recall: {macro_recall:.4f}\n")
    f.write(f"# timestamp: {timestamp}\n")
    f.write("# columns: word | gold_tag | predicted_tag\n")
    f.write("\n")
    for sentence in annotated_sentences:
        for word, gold, predicted in sentence:
            f.write(f"{word}\t{gold}\t{predicted}\n")
        f.write("\n")

print(f"Annotated path {annot_path}")

#save results
results = {
    "timestamp": timestamp,
    "language": LANGUAGE,
    "model": FINBERT,
    "strategy": STRAT,
    "notes": NOTES,
    "hyperparameters": {
        "learning_rate": LR,
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "max_length": MAX_LENGTH,
        "dev_size": DEV_SIZE,
        "seed": SEED,
        "early_stopping_patience": EARLY_STOPPING_PATIENCE,
        "early_stopping_threshold": EARLY_STOPPING_THRESHOLD,
        "metric_for_best_model": METRIC,
        "exclude_tags_from_eval": sorted(EXCLUDED_TAGS),
    },
    "tags":sorted(tag2id.keys()),
    "overall_accuracy": overall_accuracy,
    "overall_correct": overall_correct,
    "overall_total": overall_total,
    "macro_avg": {
        "precision": float(macro_precision),
        "recall": float(macro_recall),
        "f1": float(macro_f1),
    },
    "weighted_avg": {
        "precision": float(weighted_precision),
        "recall": float(weighted_recall),
        "f1": float(weighted_f1),
    },
    "per_tag": {
        tag: {
            "correct": tag_correct[tag],
            "total": tag_total[tag],
            "accuracy": per_tag_accuracy[tag],
            "precision": float(per_tag_precision[tag]),
            "recall": float(per_tag_recall[tag]),
            "f1": float(per_tag_f1[tag]),
        }
        for tag in sorted(tag2id.keys())
        if tag_total[tag] > 0
    },
}

with open(result_path, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print(f"Results -> {result_path}")


In [ ]:
from google.colab import files

files.download(result_path)
files.download(annot_path)